# Create Mochida Memorial Foundation Awards (持田記念医学薬学振興財団)

Creates the research grants and study-abroad grants of the Mochida Memorial Foundation
for Medical and Pharmaceutical Research (公益財団法人 持田記念医学薬学振興財団) from the
foundation's own recipient lists.

**Prerequisite:** `scripts/local/mochida_to_s3.py` (uploads the parquet).

**Source:** https://www.mochidazaidan.or.jp/r_list.html links PDF recipient lists per
year or multi-year block for 研究助成金交付対象者 (research grants) and 留学補助金交付対象者
(study-abroad grants). Each list is a table № / name / institution / theme under year
and research-area headings. Lists from 2001 on have a text layer and are parsed; the
1983-2000 lists are scanned images and are not included (no OCR). The tracker's
"bulk Excel" note was wrong: document.html only holds grantee report templates.

**2,706 awards, 2001-2025:** 2,197 research grants, 8 FY2013 30th-anniversary research
grants, 501 study-abroad grants (one '欠番' vacant entry dropped). Parsed counts equal
the grant counts in the foundation's annual business reports for every year they
exist (FY2008, FY2010-2025) and the per-section counts printed in the 2001-2008
headings. 100% title, institution and named grantee.

**Amounts (JPY):** the lists print none; the foundation's annual business reports
(事業報告, public_information.html) state one per-grant amount per programme and year:
research grant 2,000,000 (FY2008) and 3,000,000 (FY2010-2025), 30th-anniversary
research grant 10,000,000 (FY2013, over 3 years), study-abroad grant 500,000 (FY2008,
FY2010-2025); FY2001 research grants 1,000,000 per the note printed in that year's
list. Other years NULL (78% coverage).

**Scope:** both individual-researcher pathways are kept. Not included: researcher-
invitation grants to academic societies (研究者招聘助成金, conference support) and the
Mochida Memorial Prize (持田記念学術賞) - separate lists, follow-up.

**Field mapping:** `display_name` = research theme; `funder_scheme` = programme;
`funding_type` = research / fellowship (study abroad); `start_year` = fiscal year of
award; dates NULL; `lead_investigator` = grantee (family-first: first token = family;
names printed without a space stay whole in `family_name`, sumitomo precedent),
affiliation = institution as printed, country JP for domestic institutions not set.

**`funder_award_id` (2.1.1):** none is published; acknowledgement shells for this
funder mostly carry other funders' numbers (KAKEN, NIH), so a synthetic
`MOCHIDA-<RG|AG|SA>-<year>-<section>.<area>-<entry no>` key from the list's own
numbering is used; 0 shells collapse.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320323467`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 045p8e908, doi 10.13039/501100005865).
  Note: Mochida Pharmaceutical Company F4320322913 shares the same ROR id (no award rows).

**Priority:** `570` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mochida_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mochida/mochida_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(grant_year) as min_year, MAX(grant_year) as max_year, ROUND(SUM(amount), 0) as total_jpy
FROM openalex.awards.mochida_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mochida_raw;

In [ ]:
%sql
SELECT grant_year, programme, COUNT(*) AS n, ROUND(SUM(amount), 0) AS jpy
FROM openalex.awards.mochida_raw GROUP BY 1, 2 ORDER BY 1, 2;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320323467 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320323467;

## Step 2: Create Mochida Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mochida_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323467  -- Mochida Memorial Foundation
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Mochida Memorial Foundation grant: ', g.grantee, ' (', g.grant_year, ')')) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'JPY' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,
    'mochida_recipient_lists' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.grant_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.mochida_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
Synthetic `MOCHIDA-<RG|AG|SA>-<year>-<section>.<area>-<entry no>` keys only.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.mochida_awards
WHERE NOT funder_award_id RLIKE '^MOCHIDA-(RG|AG|SA)-(19|20)[0-9]{2}-[0-9]+[.][0-9]+-[0-9]{2,3}$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.mochida_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mochida_recipient_lists' AND priority = 570;

-- Priority 570: direct-from-funder ingest of the foundation's own recipient lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    570 as priority
FROM openalex.awards.mochida_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.mochida_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_jpy
FROM openalex.awards.mochida_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_jpy
FROM openalex.awards.mochida_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.mochida_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.mochida_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.mochida_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mochida_recipient_lists'
GROUP BY provenance, priority;